In [1]:
# HealthGuard AI - Heart Disease Data Cleaning
# Author: HealthGuard AI Team
# Date: 2026

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
%matplotlib inline

plt.style.use('seaborn-v0_8')

print("=" * 50)
print("  HealthGuard AI - Heart Disease Cleaning")
print("=" * 50)
print("Libraries Loaded Successfully!")

  HealthGuard AI - Heart Disease Cleaning
Libraries Loaded Successfully!


In [2]:
# Load Raw Heart Disease Dataset

df = pd.read_csv("E:/HealthGuard_AI/data/raw/heart.csv")

print(f"Original Shape: {df.shape}")
print(f"Total Missing Values: {df.isnull().sum().sum()}")
df.head()

Original Shape: (1025, 14)
Total Missing Values: 0


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [3]:
# Step 1: Remove Duplicate Rows
# Heart dataset has many duplicates!

print("=" * 50)
print("STEP 1: REMOVE DUPLICATES")
print("=" * 50)

before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Before: {before} rows")
print(f"After: {after} rows")
print(f"Removed: {before - after} duplicates")

STEP 1: REMOVE DUPLICATES
Before: 1025 rows
After: 302 rows
Removed: 723 duplicates


In [4]:
# Step 2: Handle Missing Values

print("=" * 50)
print("STEP 2: HANDLE MISSING VALUES")
print("=" * 50)

print("Missing Values:")
print(df.isnull().sum())
total = df.isnull().sum().sum()
print(f"\nTotal Missing: {total}")

if total == 0:
    print("No Missing Values!")
else:
    # Fill numeric with median
    numeric_cols = df.select_dtypes(
        include=['float64', 'int64']).columns
    df[numeric_cols] = df[numeric_cols].fillna(
        df[numeric_cols].median())
    print("Missing Values Fixed!")

STEP 2: HANDLE MISSING VALUES
Missing Values:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          0
thal        0
target      0
dtype: int64

Total Missing: 0
No Missing Values!


In [5]:
# Step 3: Remove Outliers Using IQR Method

print("=" * 50)
print("STEP 3: REMOVE OUTLIERS (IQR METHOD)")
print("=" * 50)

before = len(df)

outlier_cols = ['trestbps', 'chol', 'thalach', 'oldpeak']

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {outliers} outliers removed")

    df = df[(df[col] >= lower) & (df[col] <= upper)]

after = len(df)
print(f"\nBefore: {before} rows")
print(f"After: {after} rows")
print(f"Total Removed: {before - after}")

STEP 3: REMOVE OUTLIERS (IQR METHOD)
trestbps: 9 outliers removed
chol: 5 outliers removed
thalach: 1 outliers removed
oldpeak: 4 outliers removed

Before: 302 rows
After: 283 rows
Total Removed: 19


In [6]:
# Step 4: Feature Scaling

print("=" * 50)
print("STEP 4: FEATURE SCALING")
print("=" * 50)

X = df.drop('target', axis=1)
y = df['target']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Scaling Complete!")
print(f"Features Shape: {X_scaled.shape}")

STEP 4: FEATURE SCALING
Scaling Complete!
Features Shape: (283, 13)


In [7]:
# Step 5+6: Train Test Split FIRST, then SMOTE only on Training data

print("=" * 50)
print("STEP 5: TRAIN TEST SPLIT (BEFORE SMOTE)")
print("=" * 50)

X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

print(f"Train (before SMOTE): {len(X_train_raw)}")
print(f"Test (untouched): {len(X_test)}")

print("\n" + "=" * 50)
print("STEP 6: SMOTE ON TRAINING DATA ONLY")
print("=" * 50)

print("Before SMOTE:")
print(f"No Disease (0): {(y_train_raw==0).sum()}")
print(f"Disease (1): {(y_train_raw==1).sum()}")

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)

print("\nAfter SMOTE:")
print(f"No Disease (0): {(y_train==0).sum()}")
print(f"Disease (1): {(y_train==1).sum()}")
print(f"Final Training: {len(X_train)}, Final Testing: {len(X_test)}")

STEP 5: TRAIN TEST SPLIT (BEFORE SMOTE)
Train (before SMOTE): 226
Test (untouched): 57

STEP 6: SMOTE ON TRAINING DATA ONLY
Before SMOTE:
No Disease (0): 100
Disease (1): 126

After SMOTE:
No Disease (0): 126
Disease (1): 126
Final Training: 252, Final Testing: 57


In [8]:
# Step 7: Save All Cleaned Data

print("=" * 50)
print("STEP 7: SAVE CLEANED DATA")
print("=" * 50)

df_cleaned = pd.concat([X_scaled,
                        y.reset_index(drop=True)], axis=1)
df_cleaned.to_csv(
    "E:/HealthGuard_AI/data/processed/heart_cleaned_final.csv",
    index=False)

X_train.to_csv(
    "E:/HealthGuard_AI/data/processed/heart_X_train.csv",
    index=False)
X_test.to_csv(
    "E:/HealthGuard_AI/data/processed/heart_X_test.csv",
    index=False)
y_train.to_csv(
    "E:/HealthGuard_AI/data/processed/heart_y_train.csv",
    index=False)
y_test.to_csv(
    "E:/HealthGuard_AI/data/processed/heart_y_test.csv",
    index=False)

print("Files Saved:")
print(" heart_cleaned_final.csv")
print(" heart_X_train.csv")
print(" heart_X_test.csv")
print(" heart_y_train.csv")
print(" heart_y_test.csv")

STEP 7: SAVE CLEANED DATA
Files Saved:
 heart_cleaned_final.csv
 heart_X_train.csv
 heart_X_test.csv
 heart_y_train.csv
 heart_y_test.csv


In [9]:
# Cleaning Summary

print("=" * 60)
print("   HEART DISEASE CLEANING - SUMMARY REPORT")
print("=" * 60)

print("\nTECHNIQUES USED:")
print("-" * 40)
print("1. Duplicate Removal")
print("2. Missing Value Check")
print("3. IQR Outlier Removal")
print("4. Standard Scaling")
print("5. SMOTE")
print("6. Train Test Split 80/20")

print("\nRESULTS:")
print("-" * 40)
print(f"Original Rows: 1025")
print(f"After Duplicate Removal: {len(df)}")
print(f"Training Set (after SMOTE): {len(X_train)}")
print(f"Testing Set (untouched, real): {len(X_test)}")

print("\nHeart Disease Cleaning Complete!")
print("=" * 60)

   HEART DISEASE CLEANING - SUMMARY REPORT

TECHNIQUES USED:
----------------------------------------
1. Duplicate Removal
2. Missing Value Check
3. IQR Outlier Removal
4. Standard Scaling
5. SMOTE
6. Train Test Split 80/20

RESULTS:
----------------------------------------
Original Rows: 1025
After Duplicate Removal: 283
Training Set (after SMOTE): 252
Testing Set (untouched, real): 57

Heart Disease Cleaning Complete!


In [10]:
# Save Scaler for Web App
import pickle

scaler_path = "E:/HealthGuard_AI/models/saved/heart_scaler.pkl"
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(" Heart Disease Scaler Saved!")
print(f"Location: {scaler_path}")

 Heart Disease Scaler Saved!
Location: E:/HealthGuard_AI/models/saved/heart_scaler.pkl
